# Green-channel Cellpose ROI trace extraction

This notebook:
1. Loads green-channel Cellpose masks from `*_seg.npy`
2. Extracts fluorescence traces from the Suite2p registered green binary
3. Calculates ΔF/F
4. Saves `F.npy`, `F0.npy`, `dff.npy`, `stat.npy`, `iscell.npy`, and `roi_table.csv`
5. Plots ΔF/F traces with one ROI per row


## Session 216, run 04 — MY_71_R
This is a separate copy of green_cellpose_extract_traces_and_plot_dff.ipynb. All input paths target session 216, run 04. Run cells in order in the mini2p-preprocessing environment. Saved output folders are specific to this workflow within that run. Existing output files in those folders are replaced when their save cells run. Trace-display ranges default to 2,000 frames where configurable; calculations use the whole recording.


In [1]:
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt


## 1. Set paths and parameters

In [2]:
# ----------------------------
# Paths
# ----------------------------
PLANE_DIR = Path('/Volumes/behrens/meg/3x3_field_magnitude_bandit/derivatives/cohort-01/sub-03_id-MY_71_R/ses-216_date-20260730/funcimg/run-04/suite2p/plane0')


# Green Cellpose segmentation file
GREEN_SEG = PLANE_DIR / 'meanImg_green_functional_contrast_seg.npy'

# Registered green binary from Suite2p
# Usually this is data.bin. If your green channel is data_chan2.bin, change it here.
GREEN_BIN = PLANE_DIR / 'data.bin'

# Output folder
OUT_DIR = PLANE_DIR / 'custom_green_mask_traces'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Extraction settings
CHUNK_SIZE = 100
BASELINE_PERCENTILE = 20.0

print('Plane dir:', PLANE_DIR)
print('Green seg:', GREEN_SEG)
print('Green binary:', GREEN_BIN)
print('Output dir:', OUT_DIR)


Plane dir: /Volumes/behrens/meg/3x3_field_magnitude_bandit/derivatives/cohort-01/sub-03_id-MY_71_R/ses-216_date-20260730/funcimg/run-04/suite2p/plane0
Green seg: /Volumes/behrens/meg/3x3_field_magnitude_bandit/derivatives/cohort-01/sub-03_id-MY_71_R/ses-216_date-20260730/funcimg/run-04/suite2p/plane0/meanImg_green_functional_contrast_seg.npy
Green binary: /Volumes/behrens/meg/3x3_field_magnitude_bandit/derivatives/cohort-01/sub-03_id-MY_71_R/ses-216_date-20260730/funcimg/run-04/suite2p/plane0/data.bin
Output dir: /Volumes/behrens/meg/3x3_field_magnitude_bandit/derivatives/cohort-01/sub-03_id-MY_71_R/ses-216_date-20260730/funcimg/run-04/suite2p/plane0/custom_green_mask_traces


## 2. Define helper functions

In [3]:
def load_cellpose_masks(seg_path):
    """Load Cellpose masks from a *_seg.npy file."""
    seg = np.load(seg_path, allow_pickle=True).item()

    if 'masks' not in seg:
        raise KeyError(f"No 'masks' array found in {seg_path}")

    masks = np.asarray(seg['masks'])

    if masks.ndim != 2:
        raise ValueError(f'Expected 2D masks in {seg_path}, got shape {masks.shape}')

    return masks


def masks_to_rois(mask_image):
    """Convert a 2D labeled mask image into a list of Suite2p-style ROI dicts."""
    rois = []

    mask_ids = sorted(int(v) for v in np.unique(mask_image) if v > 0)

    for mask_id in mask_ids:
        ypix, xpix = np.nonzero(mask_image == mask_id)

        if len(ypix) == 0:
            continue

        rois.append(
            {
                'source_mask_id': mask_id,
                'ypix': ypix.astype(np.int64),
                'xpix': xpix.astype(np.int64),
                'lam': np.ones(len(ypix), dtype=np.float32),
                'npix': int(len(ypix)),
                'med': np.array([np.median(ypix), np.median(xpix)], dtype=np.float32),
            }
        )

    return rois


def suite2p_stat_from_rois(rois):
    """Create a Suite2p-compatible stat.npy object array from custom ROIs."""
    stat = []

    for roi in rois:
        stat.append(
            {
                'ypix': roi['ypix'],
                'xpix': roi['xpix'],
                'lam': roi['lam'],
                'npix': roi['npix'],
                'med': roi['med'],
                'source_mask_id': roi['source_mask_id'],
            }
        )

    return np.asarray(stat, dtype=object)


def extract_f_traces(movie, rois, chunk_size=500):
    """Extract mean fluorescence for each ROI from a memmapped movie."""
    n_frames = movie.shape[0]
    f = np.zeros((len(rois), n_frames), dtype=np.float32)

    for start in range(0, n_frames, chunk_size):
        stop = min(start + chunk_size, n_frames)
        chunk = movie[start:stop]

        for roi_i, roi in enumerate(rois):
            if len(roi['ypix']) == 0:
                f[roi_i, start:stop] = np.nan
                continue
            pixels = chunk[:, roi['ypix'], roi['xpix']]
            f[roi_i, start:stop] = pixels.mean(axis=1)

        print(f'Extracted {stop}/{n_frames} frames')

    return f


def make_neuropil_ring_rois(mask_image, rois, inner_radius=3, outer_radius=15):
    """Create an annular neuropil ROI around each cell, excluding all cells."""
    from scipy.ndimage import binary_dilation

    if inner_radius < 0 or outer_radius <= inner_radius:
        raise ValueError('Require 0 <= inner_radius < outer_radius.')

    occupied = np.asarray(mask_image) > 0
    neuropil_rois = []
    for roi in rois:
        cell_mask = np.zeros(occupied.shape, dtype=bool)
        cell_mask[roi['ypix'], roi['xpix']] = True
        inner = binary_dilation(cell_mask, iterations=inner_radius)
        outer = binary_dilation(cell_mask, iterations=outer_radius)
        ring = outer & ~inner & ~occupied
        ypix, xpix = np.nonzero(ring)
        neuropil_rois.append({
            'ypix': ypix.astype(np.int64),
            'xpix': xpix.astype(np.int64),
            'npix': int(len(ypix)),
        })
    return neuropil_rois


def calculate_dff(f, baseline_percentile=20.0):
    """Calculate dF/F using a percentile baseline per ROI."""
    f0 = np.percentile(f, baseline_percentile, axis=1, keepdims=True)

    # Never replace a non-positive baseline with 1e-6: doing so turns
    # ordinary signed fluorescence into billion-scale numerical artifacts.
    invalid = (~np.isfinite(f0[:, 0])) | (f0[:, 0] <= 1e-6)
    valid = ~invalid
    dff = np.full(f.shape, np.nan, dtype=np.float32)
    dff[valid] = ((f[valid] - f0[valid]) / f0[valid]).astype(np.float32)
    if np.any(invalid):
        bad = np.flatnonzero(invalid)
        print(
            f'WARNING: skipping {len(bad)} ROIs with non-positive/invalid baselines: '
            f'{bad.tolist()}'
        )

    return f0[:, 0], dff


def save_roi_table(path, rois):
    """Save a CSV table describing each ROI."""
    with path.open('w', newline='') as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=['roi_id', 'source_mask_id', 'npix', 'med_y', 'med_x'],
        )

        writer.writeheader()

        for roi_id, roi in enumerate(rois):
            writer.writerow(
                {
                    'roi_id': roi_id,
                    'source_mask_id': roi['source_mask_id'],
                    'npix': roi['npix'],
                    'med_y': float(roi['med'][0]),
                    'med_x': float(roi['med'][1]),
                }
            )


## 3. Load Suite2p metadata and Cellpose masks

In [4]:
# Load Suite2p ops
ops = np.load(PLANE_DIR / 'ops.npy', allow_pickle=True).item()

ly = int(ops['Ly'])
lx = int(ops['Lx'])

print('Suite2p frame shape:', (ly, lx))

# Load green Cellpose masks
green_masks = load_cellpose_masks(GREEN_SEG)

print('Green mask shape:', green_masks.shape)
print('Number of green Cellpose mask IDs:', int(green_masks.max()))

if green_masks.shape != (ly, lx):
    raise ValueError(
        f'Green masks shape {green_masks.shape} does not match Suite2p frame shape {(ly, lx)}'
    )

# Convert masks to ROIs
rois = masks_to_rois(green_masks)

if not rois:
    raise ValueError('No ROIs found in green Cellpose masks')

print('ROIs to extract:', len(rois))


Suite2p frame shape: (478, 501)
Green mask shape: (478, 501)
Number of green Cellpose mask IDs: 400
ROIs to extract: 400


## 4. Load registered green binary

In [5]:
# Calculate number of frames from file size
n_frames = GREEN_BIN.stat().st_size // (
    np.dtype('int16').itemsize * ly * lx
)

if n_frames == 0:
    raise ValueError(f'No frames found in {GREEN_BIN}')

movie = np.memmap(
    GREEN_BIN,
    dtype='int16',
    mode='r',
    shape=(n_frames, ly, lx),
)

print('Green binary:', GREEN_BIN)
print('Movie shape:', movie.shape)


Green binary: /Volumes/behrens/meg/3x3_field_magnitude_bandit/derivatives/cohort-01/sub-03_id-MY_71_R/ses-216_date-20260730/funcimg/run-04/suite2p/plane0/data.bin
Movie shape: (8503, 478, 501)


## 5. Extract fluorescence traces

In [8]:
f = extract_f_traces(
    movie,
    rois,
    chunk_size=1000,
)

print('F shape:', f.shape)


KeyboardInterrupt: 

## 6. Calculate ΔF/F

In [ ]:
f0, dff = calculate_dff(
    f,
    baseline_percentile=BASELINE_PERCENTILE,
)

print('F0 shape:', f0.shape)
print('dF/F shape:', dff.shape)
print('dF/F range:', np.nanmin(dff), np.nanmax(dff))


NameError: name 'f' is not defined

### Alternative: sliding-window baseline ΔF/F

In [ ]:
from scipy.ndimage import percentile_filter

# Neuropil and sliding-baseline input parameters.
NEUROPIL_INNER_RADIUS_PIXELS = 3
NEUROPIL_OUTER_RADIUS_PIXELS = 15
NEUROPIL_COEFFICIENT = 0.7
SLIDING_FRAME_RATE_HZ = 7.52
SLIDING_WINDOW_SECONDS = 30.0
SLIDING_BASELINE_PERCENTILE = 20.0

# Build a ring around each cell, excluding the cell itself and every
# other segmented ROI, then extract its average fluorescence.
neuropil_rois = make_neuropil_ring_rois(
    green_masks,
    rois,
    inner_radius=NEUROPIL_INNER_RADIUS_PIXELS,
    outer_radius=NEUROPIL_OUTER_RADIUS_PIXELS,
)
f_neuropil = extract_f_traces(movie, neuropil_rois, chunk_size=CHUNK_SIZE)
neuropil_pixel_counts = np.array([roi['npix'] for roi in neuropil_rois])

# Manual neuropil correction: Fcorrected = Fcell - 0.7 * Fneuropil.
f_corrected = (
    np.asarray(f, dtype=np.float32)
    - NEUROPIL_COEFFICIENT * f_neuropil
).astype(np.float32)

sliding_window_frames = int(round(
    SLIDING_FRAME_RATE_HZ * SLIDING_WINDOW_SECONDS
))
if sliding_window_frames < 1:
    raise ValueError('The sliding baseline window must contain at least one frame.')

# Calculate a centred, time-varying percentile baseline independently
# for every ROI. mode='nearest' handles the recording boundaries.
f0_sliding_neuropil = percentile_filter(
    f_corrected,
    percentile=SLIDING_BASELINE_PERCENTILE,
    size=(1, sliding_window_frames),
    mode='nearest',
)

valid_sliding_baseline = (
    np.isfinite(f0_sliding_neuropil) & (f0_sliding_neuropil > 1e-6)
)
dff_sliding_neuropil = np.full(f.shape, np.nan, dtype=np.float32)
np.divide(
    f_corrected - f0_sliding_neuropil,
    f0_sliding_neuropil,
    out=dff_sliding_neuropil,
    where=valid_sliding_baseline,
)

valid_sliding_rois = np.any(np.isfinite(dff_sliding_neuropil), axis=1)

# Preserve the original calculation for comparison, then make the
# neuropil-corrected sliding result the default for every later cell.
f0_constant = f0
dff_constant = dff
f0 = f0_sliding_neuropil
dff = dff_sliding_neuropil
print(
    f'Sliding baseline: {sliding_window_frames} frames '
    f'({SLIDING_WINDOW_SECONDS:g} s at {SLIDING_FRAME_RATE_HZ:g} Hz), '
    f'{SLIDING_BASELINE_PERCENTILE:g}th percentile'
)
print(f'Valid sliding-window ROIs: {valid_sliding_rois.sum()} / {len(valid_sliding_rois)}')
print(
    f'Neuropil rings: median {np.median(neuropil_pixel_counts):.0f} pixels; '
    f'coefficient {NEUROPIL_COEFFICIENT:g}'
)
print('Sliding neuropil-corrected F0 shape:', f0.shape)
print('Sliding neuropil-corrected ΔF/F shape:', dff.shape)
print('All subsequent cells now use this result through the dff variable.')


## 7. Save outputs

In [ ]:
# Canonical files now contain the neuropil-corrected sliding-baseline path
# used by the behavior-alignment notebook. Comparison intermediates are
# saved under explicit names.
np.save(OUT_DIR / 'F.npy', f_corrected)
np.save(OUT_DIR / 'F0.npy', f0)
np.save(OUT_DIR / 'dff.npy', dff)
np.save(OUT_DIR / 'F_cell.npy', f)
np.save(OUT_DIR / 'F_neuropil.npy', f_neuropil)
np.save(OUT_DIR / 'F_corrected.npy', f_corrected)
np.save(OUT_DIR / 'F0_constant.npy', f0_constant)
np.save(OUT_DIR / 'dff_constant.npy', dff_constant)
np.save(OUT_DIR / 'F0_sliding_neuropil.npy', f0_sliding_neuropil)
np.save(OUT_DIR / 'dff_sliding_neuropil.npy', dff_sliding_neuropil)
np.save(OUT_DIR / 'neuropil_pixel_counts.npy', neuropil_pixel_counts)
np.save(OUT_DIR / 'stat.npy', suite2p_stat_from_rois(rois))
valid_dff_rois = np.any(np.isfinite(dff), axis=1)
np.save(OUT_DIR / 'valid_dff_rois.npy', valid_dff_rois)
np.save(OUT_DIR / 'iscell.npy', np.column_stack([valid_dff_rois.astype(float), valid_dff_rois.astype(float)]))

save_roi_table(OUT_DIR / 'roi_table.csv', rois)

print('Saved outputs to:', OUT_DIR)


## 8. Plot one ROI

In [ ]:
roi = 1

plt.figure(figsize=(14, 4))
plt.plot(dff[roi], lw=0.8)
plt.xlabel('Frame')
plt.ylim(-3,6)
plt.ylabel('ΔF/F')
plt.title(f'ROI {roi} ΔF/F')
plt.tight_layout()
plt.show()


## 9. Plot 30 ROIs on separate rows

In [ ]:
valid_roi_indices = np.flatnonzero(np.any(np.isfinite(dff), axis=1))
rois_to_plot = valid_roi_indices[:30]

LOW_PCTL = 1
HIGH_PCTL = 95

fig, axes = plt.subplots(
    len(rois_to_plot),
    1,
    figsize=(15, 1.5 * len(rois_to_plot)),
    sharex=True,
)

if len(rois_to_plot) == 1:
    axes = [axes]

for ax, roi in zip(axes, rois_to_plot):
    trace = dff[roi]

    ax.plot(trace, lw=0.7)

    ymin = np.percentile(trace, LOW_PCTL)
    ymax = np.percentile(trace, HIGH_PCTL)

    pad = 0.1 * (ymax - ymin)
    ax.set_ylim(ymin - pad, ymax + pad)

    ax.set_ylabel(str(roi), rotation=0, labelpad=20)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(0,2500)

axes[-1].set_xlabel("Frame")
plt.suptitle("ΔF/F traces, percentile-scaled")
plt.tight_layout()
plt.show()


### Stacked ΔF/F traces

In [ ]:
# Plot multiple valid ROIs on one axis with vertical offsets.
N_STACKED_ROIS = 20
STACKED_START_FRAME = 0
STACKED_END_FRAME = 2000  # use None for the complete recording
STACKED_LOW_PCTL = 1
STACKED_HIGH_PCTL = 95
STACK_SPACING = 1.5

valid_roi_indices = np.flatnonzero(np.any(np.isfinite(dff_sliding), axis=1))
stacked_rois = valid_roi_indices[:N_STACKED_ROIS]
frame_slice = slice(STACKED_START_FRAME, STACKED_END_FRAME)
frame_numbers = np.arange(dff.shape[1])[frame_slice]
offsets = np.arange(len(stacked_rois)) * STACK_SPACING
colors = plt.cm.tab10(np.arange(len(stacked_rois)) % 10)

fig, ax = plt.subplots(figsize=(16, max(6, 0.45 * len(stacked_rois))))
for offset, color, roi in zip(offsets, colors, stacked_rois):
    trace = np.asarray(dff[roi, frame_slice], dtype=float)
    finite = np.isfinite(trace)
    if finite.sum() < 2:
        continue
    low, high = np.nanpercentile(trace, [STACKED_LOW_PCTL, STACKED_HIGH_PCTL])
    scale = high - low
    if not np.isfinite(scale) or scale <= 0:
        continue
    # Robustly put every ROI on a comparable visual scale. Clipping is
    # display-only and does not change dff.
    scaled = (trace - low) / scale
    scaled = np.clip(scaled, -0.25, 1.25)
    ax.plot(frame_numbers, scaled + offset, color=color, lw=0.65)

ax.set_yticks(offsets + 0.5, [f'ROI {roi}' for roi in stacked_rois])
ax.set_xlabel('Imaging frame')
ax.set_ylabel('ROI')
ax.set_title(f'First {len(stacked_rois)} valid ΔF/F traces (stacked)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.margins(x=0.01, y=0.03)
fig.tight_layout()
plt.show()


## 10. Optional: plot with time in seconds

In [ ]:
# Set your imaging frame rate here if you know it
FS = 7.52  # Hz; change this if your frame rate is different

time = np.arange(dff.shape[1]) / FS
valid_roi_indices = np.flatnonzero(np.any(np.isfinite(dff), axis=1))
rois_to_plot = valid_roi_indices[:30]

fig, axes = plt.subplots(
    len(rois_to_plot),
    1,
    figsize=(15, 1.5 * len(rois_to_plot)),
    sharex=True,
)

if len(rois_to_plot) == 1:
    axes = [axes]

for ax, roi in zip(axes, rois_to_plot):
    ax.plot(time, dff[roi], lw=0.7)
    ax.set_ylabel(str(roi), rotation=0, labelpad=20)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('ΔF/F traces')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation


# ---------------------------------------------------------
# Settings
# ---------------------------------------------------------
N_CELLS_TO_PLOT = 20

# A cell is considered to show a spike when its frame-to-frame
# change exceeds this many robust standard deviations.
CELL_SPIKE_THRESHOLD =  5

# Choose 'derivative_z' for the hard robust-z cutoff above, or
# 'percentile' for values outside each cell's amplitude percentiles.
ARTIFACT_DETECTION_METHOD = 'percentile'  # or 'percentile'
LOW_ARTIFACT_PERCENTILE = 1
HIGH_ARTIFACT_PERCENTILE = 95

# Mark a frame as a global artifact when this fraction of cells
# spike simultaneously.
MIN_FRACTION_CELLS = 0.5

# Also remove frames immediately around the detected artifact.
ARTIFACT_PADDING_FRAMES = 1

# Optional frame rate for plotting time in seconds.
# Set to None to plot frame number instead.
FRAME_RATE = None
# Example:
# FRAME_RATE = 30


# ---------------------------------------------------------
# Detect global spike artifacts
# ---------------------------------------------------------
def detect_global_spike_frames(
    traces,
    cell_spike_threshold=6,
    min_fraction_cells=0.25,
    padding_frames=1,
    method='derivative_z',
    low_percentile=1,
    high_percentile=99,
):
    """
    Detect frames containing sudden changes shared across many cells.

    Parameters
    ----------
    traces : ndarray
        Array with shape (n_cells, n_frames).

    cell_spike_threshold : float
        Robust z-score threshold applied to the frame-to-frame
        difference of each cell.

    min_fraction_cells : float
        Minimum fraction of cells that must cross the threshold
        for a frame to be called a global artifact.

    padding_frames : int
        Number of adjacent frames to include on each side.

    Returns
    -------
    artifact_frames : ndarray of bool
        Boolean vector with one value per frame.

    fraction_spiking : ndarray
        Fraction of cells showing an abrupt change at each frame.

    derivative_z : ndarray
        Robustly normalized frame-to-frame changes.
    """
    traces = np.asarray(traces, dtype=np.float64)

    if traces.ndim != 2:
        raise ValueError(
            "traces must have shape (n_cells, n_frames)"
        )

    # Difference between consecutive frames.
    derivative = np.diff(
        traces,
        axis=1,
        prepend=traces[:, :1],
    )

    # Robust normalization independently for each cell.
    cell_median = np.nanmedian(
        derivative,
        axis=1,
        keepdims=True,
    )

    cell_mad = np.nanmedian(
        np.abs(derivative - cell_median),
        axis=1,
        keepdims=True,
    )

    # Convert MAD to an estimate of standard deviation.
    robust_scale = 1.4826 * cell_mad

    # Avoid division by zero for flat traces.
    robust_scale[robust_scale < 1e-12] = np.nan

    derivative_z = (
        derivative - cell_median
    ) / robust_scale

    if method == 'derivative_z':
        # Hard cutoff on robustly normalized frame-to-frame changes.
        positive_spikes = derivative_z >= cell_spike_threshold
        negative_spikes = derivative_z <= -cell_spike_threshold
    elif method == 'percentile':
        # Cell-specific amplitude limits in both directions.
        low = np.nanpercentile(traces, low_percentile, axis=1, keepdims=True)
        high = np.nanpercentile(traces, high_percentile, axis=1, keepdims=True)
        positive_spikes = traces > high
        negative_spikes = traces < low
    else:
        raise ValueError("method must be 'derivative_z' or 'percentile'")
    cells_spiking = positive_spikes | negative_spikes

    valid_cells = np.any(np.isfinite(traces), axis=1)
    fraction_spiking = np.mean(
        cells_spiking[valid_cells],
        axis=0,
    )

    artifact_frames = (
        fraction_spiking >= min_fraction_cells
    )

    if padding_frames > 0:
        structure = np.ones(
            2 * padding_frames + 1,
            dtype=bool,
        )

        artifact_frames = binary_dilation(
            artifact_frames,
            structure=structure,
        )

    return (
        artifact_frames,
        fraction_spiking,
        derivative_z,
    )


# ---------------------------------------------------------
# Replace artifact frames by interpolation
# ---------------------------------------------------------
def interpolate_artifact_frames(
    traces,
    artifact_frames,
):
    """
    Replace marked frames independently in each trace using
    linear interpolation between surrounding valid frames.
    """
    traces = np.asarray(
        traces,
        dtype=np.float64,
    )

    artifact_frames = np.asarray(
        artifact_frames,
        dtype=bool,
    )

    if traces.shape[1] != len(artifact_frames):
        raise ValueError(
            "artifact_frames length must equal the number "
            "of frames in traces"
        )

    corrected = traces.copy()

    frame_numbers = np.arange(
        traces.shape[1]
    )

    valid_frames = ~artifact_frames

    if valid_frames.sum() < 2:
        raise ValueError(
            "Too few valid frames remain for interpolation."
        )

    for cell_index in range(traces.shape[0]):
        trace = traces[cell_index]

        finite_valid = (
            valid_frames
            & np.isfinite(trace)
        )

        if finite_valid.sum() < 2:
            corrected[cell_index] = trace
            continue

        corrected[
            cell_index,
            artifact_frames,
        ] = np.interp(
            frame_numbers[artifact_frames],
            frame_numbers[finite_valid],
            trace[finite_valid],
        )

    return corrected


In [ ]:
artifact_frames, fraction_spiking, derivative_z = (
    detect_global_spike_frames(
        dff,
        cell_spike_threshold=CELL_SPIKE_THRESHOLD,
        min_fraction_cells=MIN_FRACTION_CELLS,
        padding_frames=ARTIFACT_PADDING_FRAMES,
        method=ARTIFACT_DETECTION_METHOD,
        low_percentile=LOW_ARTIFACT_PERCENTILE,
        high_percentile=HIGH_ARTIFACT_PERCENTILE,
    )
)

dff_corrected = interpolate_artifact_frames(
    dff,
    artifact_frames,
)

print(f'Artifact detection method: {ARTIFACT_DETECTION_METHOD}')
print(
    f"Detected {artifact_frames.sum()} artifact frames "
    f"out of {dff.shape[1]} total frames."
)

print(
    f"Detected artifact percentage: "
    f"{100 * artifact_frames.mean():.3f}%"
)


### Stacked ΔF/F traces after artifact removal (original ROI order)

In [ ]:
# Plot the first valid corrected ROIs without responsiveness ranking.
N_CORRECTED_STACKED_ROIS = 20
CORRECTED_START_FRAME = 0
CORRECTED_END_FRAME = 2000  # use None for the complete recording
CORRECTED_LOW_PCTL = 1
CORRECTED_HIGH_PCTL = 95
CORRECTED_STACK_SPACING = 1.5

corrected_valid_rois = np.flatnonzero(np.any(np.isfinite(dff_corrected), axis=1))
corrected_stacked_rois = corrected_valid_rois[:N_CORRECTED_STACKED_ROIS]
corrected_frame_slice = slice(CORRECTED_START_FRAME, CORRECTED_END_FRAME)
corrected_frame_numbers = np.arange(dff_corrected.shape[1])[corrected_frame_slice]
corrected_offsets = np.arange(len(corrected_stacked_rois)) * CORRECTED_STACK_SPACING
corrected_colors = plt.cm.tab10(np.arange(len(corrected_stacked_rois)) % 10)

fig, ax = plt.subplots(figsize=(16, max(6, 0.45 * len(corrected_stacked_rois))))
for offset, color, roi in zip(corrected_offsets, corrected_colors, corrected_stacked_rois):
    trace = np.asarray(dff_corrected[roi, corrected_frame_slice], dtype=float)
    finite = np.isfinite(trace)
    if finite.sum() < 2:
        continue
    low, high = np.nanpercentile(trace, [CORRECTED_LOW_PCTL, CORRECTED_HIGH_PCTL])
    scale = high - low
    if not np.isfinite(scale) or scale <= 0:
        continue
    scaled = (trace - low) / scale
    scaled = np.clip(scaled, -0.25, 1.25)  # display only
    ax.plot(corrected_frame_numbers, scaled + offset, color=color, lw=0.65)

ax.set_yticks(
    corrected_offsets + 0.5,
    [f'ROI {roi}' for roi in corrected_stacked_rois],
)
ax.set_xlabel('Imaging frame')
ax.set_ylabel('ROI')
ax.set_title(
    f'First {len(corrected_stacked_rois)} valid ΔF/F traces after artifact removal'
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.margins(x=0.01, y=0.03)
fig.tight_layout()
plt.show()


In [ ]:
if FRAME_RATE is None:
    x = np.arange(dff.shape[1])
    x_label = "Frame"
else:
    x = np.arange(dff.shape[1]) / FRAME_RATE
    x_label = "Time (s)"

plt.figure(figsize=(14, 4))

plt.plot(
    x,
    fraction_spiking,
    linewidth=0.8,
)

plt.axhline(
    MIN_FRACTION_CELLS,
    linestyle="--",
    label="Artifact threshold",
)

artifact_indices = np.where(
    artifact_frames
)[0]

if len(artifact_indices) > 0:
    plt.scatter(
        x[artifact_indices],
        fraction_spiking[artifact_indices],
        s=8,
        label="Removed frames",
    )

plt.xlabel(x_label)
plt.ylabel("Fraction of cells with abrupt change")
plt.title("Detection of simultaneous population-wide spikes")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Rank by a robust high percentile rather than a single residual maximum.
# This prevents one remaining spike from dominating cell selection.
response_score = np.full(dff_corrected.shape[0], np.nan)
valid_trace_cells = np.flatnonzero(np.any(np.isfinite(dff_corrected), axis=1))
response_score[valid_trace_cells] = np.nanpercentile(
    dff_corrected[valid_trace_cells],
    HIGH_PCTL,
    axis=1,
)

valid_cells = np.where(
    np.isfinite(response_score)
)[0]

ranked_valid_cells = valid_cells[
    np.argsort(
        response_score[valid_cells]
    )[::-1]
]

top_cell_indices = ranked_valid_cells[
    :N_CELLS_TO_PLOT
]

print("Top cell indices:", top_cell_indices)

for rank, cell_index in enumerate(
    top_cell_indices,
    start=1,
):
    print(
        f"{rank:2d}. Cell {cell_index}: "
        f"corrected {HIGH_PCTL}th-percentile ΔF/F = "
        f"{response_score[cell_index]:.3f}"
    )


In [ ]:
fig, axes = plt.subplots(
    N_CELLS_TO_PLOT,
    1,
    figsize=(15, 30),
    sharex=True,
)

if N_CELLS_TO_PLOT == 1:
    axes = [axes]

for ax, cell_index in zip(
    axes,
    top_cell_indices,
):
    ax.plot(
        x,
        dff_corrected[cell_index],
        linewidth=0.8,
    )

    # Use the same percentile scaling as the earlier trace plot so calcium
    # kernels remain visible despite any residual extreme samples.
    finite_trace = dff_corrected[cell_index, np.isfinite(dff_corrected[cell_index])]
    ymin, ymax = np.percentile(finite_trace, [LOW_PCTL, HIGH_PCTL])
    pad = 0.1 * (ymax - ymin)
    ax.set_ylim(ymin - pad, ymax + pad)

    ax.set_ylabel(
        f"ROI {cell_index}\nΔF/F",
        rotation=0,
        ha="right",
        va="center",
    )

    ax.set_title(
        f"Corrected {HIGH_PCTL}th-percentile ΔF/F: "
        f"{response_score[cell_index]:.3f}",
        loc="right",
        fontsize=8,
    )

axes[-1].set_xlabel(x_label)
axes[-1].set_xlim(0, 2500)

fig.suptitle(
    "20 most responsive cells after removal of global spike artifacts",
    y=1.001,
)

plt.tight_layout()
plt.show()
